In [1]:
import os, json, time, ast, traceback
from pathlib import Path
import nbformat
import pandas as pd
from IPython.display import display
from IPython.core.interactiveshell import InteractiveShell
from IPython.utils.capture import capture_output

In [2]:
ROOT = Path.cwd()
STAGES = ['a_import', 'b_train', 'c_splice', 'd_fit', 'e_infer', 'f_evaluate', 'g_report']
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
InteractiveShell.instance()
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')

In [3]:
def execute_stage(path):
    # fresh namespace per task; retain rich outputs without starting more kernels
    nb = nbformat.read(path, as_version=4)
    namespace = {'__name__': '__main__', '__file__': str(path)}
    previous = Path.cwd()
    os.chdir(path.parent)
    try:
        for count, cell in enumerate(nb.cells, 1):
            tree = ast.parse(cell.source)
            expression = tree.body.pop() if tree.body and isinstance(tree.body[-1], ast.Expr) else None
            cell.outputs, cell.execution_count = [], count
            try:
                with capture_output() as captured:
                    exec(compile(tree, str(path), 'exec'), namespace)
                    if expression is not None:
                        value = eval(compile(ast.Expression(expression.value), str(path), 'eval'), namespace)
                        if value is not None:
                            display(value)
                if captured.stdout:
                    cell.outputs.append(nbformat.v4.new_output('stream', name='stdout', text=captured.stdout))
                if captured.stderr:
                    cell.outputs.append(nbformat.v4.new_output('stream', name='stderr', text=captured.stderr))
                for item in captured.outputs:
                    cell.outputs.append(nbformat.v4.new_output('display_data', data=item.data, metadata=item.metadata))
            except Exception as error:
                cell.outputs.append(nbformat.v4.new_output('error', ename=type(error).__name__, evalue=str(error),
                    traceback=traceback.format_exc().splitlines()))
                raise
    finally:
        os.chdir(previous)
        nbformat.write(nb, path)

In [4]:
# restore symlinks if a zip extractor wrote their targets as text files
for previous, stage in zip(STAGES, STAGES[1:]):
    path = ROOT / stage / 'input'
    if not path.is_symlink():
        if path.is_dir():
            path.rmdir()
        else:
            path.unlink(missing_ok=True)
        path.symlink_to('../' + previous + '/output', target_is_directory=True)

In [5]:
timings = []
for stage in STAGES:
    path = ROOT / stage / 'src' / f'{stage}.ipynb'
    start = time.perf_counter()
    execute_stage(path)
    timings.append({'stage': stage, 'seconds': time.perf_counter() - start})
    display(pd.DataFrame(timings))
pd.DataFrame(timings).to_csv(ROOT / 'pipeline_timings.csv', index=False)

,stage,seconds
0,a_import,1.198726


,stage,seconds
0,a_import,1.198726
1,b_train,6.495310


,stage,seconds
0,a_import,1.198726
1,b_train,6.495310
2,c_splice,68.005801


,stage,seconds
0,a_import,1.198726
1,b_train,6.495310
2,c_splice,68.005801
3,d_fit,20.093860


,stage,seconds
0,a_import,1.198726
1,b_train,6.495310
2,c_splice,68.005801
3,d_fit,20.093860
4,e_infer,19.016268


,stage,seconds
0,a_import,1.198726
1,b_train,6.495310
2,c_splice,68.005801
3,d_fit,20.093860
4,e_infer,19.016268
5,f_evaluate,1.483767


,stage,seconds
0,a_import,1.198726
1,b_train,6.495310
2,c_splice,68.005801
3,d_fit,20.093860
4,e_infer,19.016268
5,f_evaluate,1.483767
6,g_report,3.880202


In [6]:
pd.read_csv(ROOT / 'g_report/output/headline_results.csv')[
    ['panel', 'markers', 'steps', 'method', 'n', 'block_accuracy', 'boundary_f1']]

,panel,markers,steps,method,n,block_accuracy,boundary_f1
0,exact,4,0,copying,24,1.000000,1.000000
1,exact,4,0,global,24,0.545139,0.000000
2,exact,4,0,nearest,24,1.000000,1.000000
3,exact,4,300,copying,24,0.517361,0.318913
4,exact,4,300,global,24,0.286458,0.000000
5,exact,4,300,nearest,24,0.505208,0.340093
6,exact,600,0,copying,24,1.000000,1.000000
7,exact,600,0,global,24,0.559028,0.000000
8,exact,600,0,nearest,24,1.000000,1.000000
9,exact,600,300,copying,24,0.526042,0.408793
